# Slide Skill Optimizer — Training Notebook

Fine-tunes a small open model (Qwen2.5-7B) to replace Claude Opus 4.6 as the
**optimizer** in the Slide Skill OpenEnv loop.

This notebook targets the current `slide_skill_env` runtime, which uses
WebSocket sessions (`/ws`) and a `hint` action interface.

**Pipeline:**
1. **Data collection** — Run N episodes against a live Slide Skill server and capture
   (prompt, DESIGN_RULES.md after step, reward) tuples.
2. **Filtering** — Keep only steps where `reward > 0`.
3. **SFT training** — Fine-tune Qwen2.5-7B-Instruct with Unsloth + TRL SFTTrainer.
4. **Save / push** — Export QLoRA weights; optionally push to HuggingFace Hub.

**Runtime:** T4 GPU is sufficient. A100 is faster.
**Estimated collection time:** ~60-150s per step × 7 steps × N episodes.


## 1. Install Dependencies

In [ ]:
# Unsloth must be installed before transformers to patch correctly.
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q anthropic httpx websockets trl datasets transformers accelerate bitsandbytes
print('Done.')


## 2. Configuration

In [ ]:
import os

# ------------------------------------------------------------------
# API keys (set here or via Colab Secrets)
# ------------------------------------------------------------------
os.environ.setdefault('ANTHROPIC_API_KEY', '')
os.environ.setdefault('GEMINI_API_KEY', '')
os.environ.setdefault('PATRONUS_API_KEY', '')

# Optional local stability if Patronus TLS fails in your environment:
# os.environ.setdefault('SLIDE_SKILL_SKIP_PATRONUS', 'true')

# ------------------------------------------------------------------
# OpenEnv server URL
# ------------------------------------------------------------------
# Option A: HuggingFace Spaces deployment
#   SERVER_URL = 'https://your-username-slide-skill-openenv.hf.space'
# Option B: Local server exposed via ngrok
#   !pip install -q pyngrok
#   from pyngrok import ngrok; ngrok.set_auth_token('YOUR_TOKEN')
#   tunnel = ngrok.connect(8000); SERVER_URL = tunnel.public_url
SERVER_URL = 'https://your-hf-space-url.hf.space'  # <-- EDIT THIS

# ------------------------------------------------------------------
# Data collection
# ------------------------------------------------------------------
N_EPISODES = 20
MAX_STEPS_PER_EPISODE = 7
MIN_REWARD = 0.0
DATA_FILE = 'trajectories.json'
REQUEST_TIMEOUT_S = 300

# ------------------------------------------------------------------
# Training
# ------------------------------------------------------------------
BASE_MODEL = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 4096
OUTPUT_DIR = './optimizer-model'
HF_REPO = ''  # Optional: 'your-username/slide-optimizer'

print(f'Server: {SERVER_URL}')
print(f'Collecting {N_EPISODES} episodes, training on steps with reward > {MIN_REWARD}')


## 3. Data Collection

In [ ]:
import asyncio
import base64
import json
import textwrap
from dataclasses import dataclass, asdict
from pathlib import Path

import websockets


In [ ]:
OPTIMIZER_SYSTEM = (
    'You are a McKinsey slide design optimizer. '
    'You improve a PowerPoint generation skill by rewriting its DESIGN_RULES.md file.'
)


def _extract_obs(resp: dict) -> dict:
    # Extract observation from OpenEnv WS response.
    data = resp['data']
    obs = data.get('observation', data)
    if 'done' in data and 'done' not in obs:
        obs['done'] = data['done']
    if 'reward' in data and 'reward' not in obs:
        obs['reward'] = data['reward']
    return obs


def build_optimizer_prompt(obs: dict) -> str:
    # Format an observation dict into optimizer input text.
    scores = obs.get('scores', {})
    strengths = '\n'.join(f'- {s}' for s in obs.get('strengths', []))
    weaknesses = '\n'.join(f'- {w}' for w in obs.get('weaknesses', []))
    skill_files = obs.get('skill_files', {})
    design_rules = skill_files.get('DESIGN_RULES.md', '')
    examples = skill_files.get('EXAMPLES.md', '')

    return textwrap.dedent(f"""\
        ## Current Score: {obs.get('total', 0)}/100

        ## Score Breakdown
        - background_layout: {scores.get('background_layout', 0)}/15
        - color_palette: {scores.get('color_palette', 0)}/15
        - typography: {scores.get('typography', 0)}/15
        - title_quality: {scores.get('title_quality', 0)}/15
        - data_presentation: {scores.get('data_presentation', 0)}/15
        - structural_elements: {scores.get('structural_elements', 0)}/15
        - overall_impression: {scores.get('overall_impression', 0)}/10

        ## Evaluator Feedback
        Strengths:
        {strengths}

        Weaknesses:
        {weaknesses}

        Verdict: {obs.get('one_line_verdict', '')}

        ## Current DESIGN_RULES.md
        {design_rules}

        ## Current EXAMPLES.md
        {examples}

        Write an improved DESIGN_RULES.md that addresses the weaknesses above \
        while preserving what works well. Focus on the lowest-scoring dimensions.
    """)


print('WS helper functions defined.')


In [ ]:
@dataclass
class TrainingStep:
    prompt: str
    completion: str
    reward: float
    total: int
    episode: int
    step: int


REFERENCE_DIR = Path('output/reference')


def _load_reference_images() -> list[str]:
    refs: list[str] = []
    if REFERENCE_DIR.exists():
        for img_path in sorted(REFERENCE_DIR.glob('*.jpg')):
            refs.append(base64.b64encode(img_path.read_bytes()).decode())
    return refs


async def collect_episode(episode_idx: int, server_url: str, max_steps: int = MAX_STEPS_PER_EPISODE) -> list[TrainingStep]:
    # Run one optimization episode via WS and return training steps.
    steps: list[TrainingStep] = []
    refs = _load_reference_images()

    ws_url = server_url.replace('http://', 'ws://').replace('https://', 'wss://').rstrip('/') + '/ws'
    async with websockets.connect(ws_url, max_size=50 * 1024 * 1024, open_timeout=20) as ws:
        await ws.send(json.dumps({
            'type': 'reset',
            'data': {
                'reference_images': refs,
                'task_prompt': (
                    'Generate a 1-slide timeline PowerPoint about Dutch Hydrogen '
                    'Strategy (2020-2035) in McKinsey consulting style.'
                ),
                'max_steps': max_steps,
            },
        }))
        resp = json.loads(await asyncio.wait_for(ws.recv(), timeout=REQUEST_TIMEOUT_S))
        if resp.get('type') == 'error':
            raise RuntimeError(resp.get('message', 'reset failed'))

        obs = _extract_obs(resp)
        print(f'  Episode {episode_idx}: baseline score={obs.get("total", 0)}/100')

        for step_idx in range(1, max_steps + 1):
            if obs.get('done', False):
                break

            prev_obs = obs
            prompt = build_optimizer_prompt(prev_obs)

            await ws.send(json.dumps({'type': 'step', 'data': {'hint': None}}))
            resp = json.loads(await asyncio.wait_for(ws.recv(), timeout=REQUEST_TIMEOUT_S))
            if resp.get('type') == 'error':
                raise RuntimeError(resp.get('message', 'step failed'))

            obs = _extract_obs(resp)
            completion = obs.get('skill_files', {}).get('DESIGN_RULES.md', '')

            steps.append(TrainingStep(
                prompt=prompt,
                completion=completion,
                reward=float(obs.get('reward', 0.0)),
                total=int(obs.get('total', 0)),
                episode=episode_idx,
                step=step_idx,
            ))
            delta = f"{steps[-1].reward*100:+.0f}pts"
            print(f'    step {step_idx}: score={obs.get("total", 0)}/100 ({delta})')

    return steps


print('Collection function defined (WS).')


In [ ]:
all_steps: list[TrainingStep] = []

for ep in range(N_EPISODES):
    print(f'\nEpisode {ep + 1}/{N_EPISODES}')
    try:
        episode_steps = asyncio.run(collect_episode(ep + 1, SERVER_URL))
        all_steps.extend(episode_steps)
    except Exception as e:
        print(f'  Episode {ep + 1} failed: {e} — skipping')
        continue

    with open(DATA_FILE, 'w') as f:
        json.dump([asdict(s) for s in all_steps], f, indent=2)
    print(f'  Saved {len(all_steps)} steps to {DATA_FILE}')

print(f'\nCollection complete: {len(all_steps)} total steps across {N_EPISODES} episodes.')


In [ ]:
import statistics

# Load from disk (in case of kernel restart).
with open(DATA_FILE) as f:
    raw = json.load(f)
all_steps = [TrainingStep(**s) for s in raw]

rewards = [s.reward for s in all_steps]
positive = [s for s in all_steps if s.reward > MIN_REWARD]

print(f'Total steps collected : {len(all_steps)}')
print(f'Mean reward           : {statistics.mean(rewards):.3f}' if rewards else 'Mean reward           : n/a')
print(f'Positive-reward steps : {len(positive)} ({100*len(positive)/max(len(all_steps),1):.0f}%)')
print(f'Training examples     : {len(positive)}')

if len(positive) < 10:
    print('\n⚠ Very few positive examples. Consider collecting more episodes or lowering MIN_REWARD.')

## 4. Fine-tuning with Unsloth + TRL

In [ ]:
from datasets import Dataset
from unsloth import FastLanguageModel

# Load model first so we can use its tokenizer for chat formatting.
print(f'Loading {BASE_MODEL} ...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
print('Model loaded.')


def format_example(step: TrainingStep) -> dict:
    """Format a training step as a chat-template string for SFT."""
    text = tokenizer.apply_chat_template(
        [
            {'role': 'system', 'content': OPTIMIZER_SYSTEM},
            {'role': 'user', 'content': step.prompt},
            {'role': 'assistant', 'content': step.completion},
        ],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}


formatted = [format_example(s) for s in positive]
dataset = Dataset.from_list(formatted).train_test_split(test_size=0.1, seed=42)

print(f'Train examples: {len(dataset["train"])}')
print(f'Eval  examples: {len(dataset["test"])}')
print('\nSample (truncated):')
print(dataset['train'][0]['text'][:500], '...')

In [ ]:
# Add LoRA adapters via Unsloth.
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',  # Unsloth's memory-efficient checkpointing
    random_state=42,
)
print('LoRA adapters attached.')

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=4,   # effective batch size = 4
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        fp16=not FastLanguageModel.is_bfloat16_supported(),
        bf16=FastLanguageModel.is_bfloat16_supported(),
        logging_steps=5,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        report_to='none',  # set to 'wandb' if you want W&B logging
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LENGTH,
        packing=True,    # Packs short examples together to fill context — improves throughput
    ),
)

print('Starting training...')
trainer.train()
print('Training complete.')

## 5. Save & Push

In [ ]:
# Save LoRA adapters (small — only the delta weights).
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Saved to {OUTPUT_DIR}/')

# Optional: merge + save full model (larger, but self-contained for deployment).
# model.save_pretrained_merged(OUTPUT_DIR + '-merged', tokenizer, save_method='merged_16bit')

# Optional: push to HuggingFace Hub.
if HF_REPO:
    model.push_to_hub(HF_REPO, token=os.environ.get('HF_TOKEN', ''))
    tokenizer.push_to_hub(HF_REPO, token=os.environ.get('HF_TOKEN', ''))
    print(f'Pushed to https://huggingface.co/{HF_REPO}')

## 6. Smoke Test — Run Trained Model

In [ ]:
# Switch model to inference mode (disables dropout, fuses LoRA for speed).
FastLanguageModel.for_inference(model)

# Use the last collected observation as a test prompt.
if not positive:
    raise RuntimeError('No positive-reward steps to test with — run data collection first.')
test_step = positive[-1]
test_prompt = test_step.prompt

inputs = tokenizer.apply_chat_template(
    [
        {'role': 'system', 'content': OPTIMIZER_SYSTEM},
        {'role': 'user', 'content': test_prompt},
    ],
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
).to(model.device)

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=1024,
    temperature=0.7,
    do_sample=True,
)

generated = tokenizer.decode(
    outputs[0][inputs.shape[1]:],
    skip_special_tokens=True,
)

print('=== Trained model output (first 1000 chars) ===')
print(generated[:1000])